# M5 Forecasting — Notebook 2: Feature Engineering

**Pipeline:** EDA → **Feature Engineering** → Forecasting → Optimization

Processes one store at a time to keep peak RAM under ~3 GB per store.
Saves each store's features to `/kaggle/working/store_cache/` so the kernel
can resume if interrupted.

> Set `N_ITEMS = 500` for fast iteration, `None` for full dataset.


In [ ]:
import sys, pathlib

SRC_ROOT = '/kaggle/input/datasets/youssefmousaaid/m5-forecast-optimize-src-files'
DATA_DIR = '/kaggle/input/competitions/m5-forecasting-accuracy/'

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f'src/ present : {(pathlib.Path(SRC_ROOT) / "src").exists()}')
print(f'DATA_DIR     : {DATA_DIR}')


In [ ]:
import shutil, pathlib
# Clear old cache so rebuilt features.py takes effect
cache_path = pathlib.Path(CACHE_DIR)
if cache_path.exists():
    shutil.rmtree(cache_path)
    print('Cache cleared')

import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')

from src.data.features import (
    build_features, load_raw, melt_sales,
    attach_calendar, attach_prices,
    build_lag_features, build_rolling_features, build_calendar_features,
    FEATURE_COLS, TARGET_COL,
)

N_ITEMS   = None   # set to 500 for fast dev run, None for full dataset
CACHE_DIR = '/kaggle/working/store_cache'

COLORS = {'green':'#48bb78','blue':'#63b3ed','orange':'#f6ad55',
          'red':'#fc8181','gray':'#718096','purple':'#b794f4'}
plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a202c',
    'axes.edgecolor':'#2d3748','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#a0aec0','ytick.color':'#a0aec0',
    'text.color':'#e2e8f0','grid.color':'#2d3748',
})
print('Imports OK ✓')


## Build Features (per-store, with cache)

In [ ]:
# Processes 10 stores one at a time — peak RAM ~3 GB per store.
# Each store is cached to CACHE_DIR so you can resume if the kernel restarts.
# Full dataset: ~15-20 min. Dev run (N_ITEMS=500): ~2 min.

df = build_features(
    DATA_DIR,
    n_items   = N_ITEMS,
    cache_dir = CACHE_DIR,
)

print(f'\nFeature matrix: {len(df):,} rows x {df.shape[1]} columns')
print(f'RAM usage      : {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
print(f'Date range     : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Items          : {df["id"].nunique():,}')
print(f'Features       : {len(FEATURE_COLS)}')
print(f'NaNs in X      : {df[FEATURE_COLS].isna().sum().sum()}')


## Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Key Feature Distributions', fontsize=13)

pairs = [
    ('lag_7',       COLORS['blue'],   'Lag-7 Sales'),
    ('rmean_28',    COLORS['green'],  'Rolling Mean-28'),
    ('sell_price',  COLORS['orange'], 'Sell Price'),
    ('price_pct_chg', COLORS['purple'], 'Price % Change'),
    ('lag_28',      COLORS['blue'],   'Lag-28 Sales'),
    ('rstd_7',      COLORS['red'],    'Rolling Std-7'),
]
for ax, (col, color, title) in zip(axes.flatten(), pairs):
    data = df[col].dropna()
    data = data[data.between(data.quantile(0.01), data.quantile(0.99))]
    ax.hist(data, bins=60, color=color, edgecolor='none', alpha=0.85)
    ax.set_title(title); ax.set_xlabel('Value'); ax.set_ylabel('Count')

plt.tight_layout(); plt.show()


## Rolling Mean vs Actual — Sample Series

In [ ]:
sample_id = df['id'].iloc[0]
sample_series = df[df['id'] == sample_id].sort_values('date').tail(200)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(sample_series['date'], sample_series['sales'],
        color=COLORS['gray'], alpha=0.6, lw=1, label='Actual')
ax.plot(sample_series['date'], sample_series['rmean_7'],
        color=COLORS['green'], lw=1.5, label='rmean_7')
ax.plot(sample_series['date'], sample_series['rmean_28'],
        color=COLORS['orange'], lw=1.5, label='rmean_28')
ax.set_title(f'Rolling Features — {sample_id}')
ax.set_xlabel('Date'); ax.set_ylabel('Units'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df[TARGET_COL], bins=50, color=COLORS['blue'], edgecolor='none', alpha=0.85)
axes[0].set_title('Sales Distribution'); axes[0].set_xlabel('Units Sold')

nonzero = df[df[TARGET_COL] > 0][TARGET_COL]
axes[1].hist(nonzero, bins=50, color=COLORS['green'], edgecolor='none', alpha=0.85, log=True)
axes[1].set_title('Non-Zero Sales (log scale)'); axes[1].set_xlabel('Units Sold')

zero_rate = (df[TARGET_COL] == 0).mean()
plt.suptitle(f'Target: {TARGET_COL}  |  Zero rate: {zero_rate:.1%}', fontsize=12)
plt.tight_layout(); plt.show()


## Save

In [ ]:
out_path = '/kaggle/working/features.parquet'
df.to_parquet(out_path, index=False)
size_gb = df.memory_usage(deep=True).sum() / 1e9
print(f'Saved  -> {out_path}')
print(f'Size   : {size_gb:.2f} GB in memory')
print()
print('Next -> 03_forecasting.ipynb')
